In [12]:
from pathlib import Path
import pandas as pd

# Resolve dataset path robustly whether the notebook runs from project root or rag/.
candidate_paths = [
    Path("kb/eng-jap.tsv"),
    Path("../kb/eng-jap.tsv"),
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError(
        "Could not find eng-jap.tsv. Checked: kb/eng-jap.tsv and ../kb/eng-jap.tsv"
    )

print(f"Using dataset: {dataset_path.resolve()}")

# TSV format: English \t Japanese
df = pd.read_csv(
    dataset_path,
    sep="\t",
    header=None,
    names=["eng_id","english","jp_id","japanese"],
    dtype=str,
).fillna("")

df.head(5)

Using dataset: C:\Users\kaile\Documents\GitHub\is469\kb\eng-jap.tsv


,eng_id,english,jp_id,japanese
0,1276,Let's try something.,4702,何かしてみましょう。
1,1277,I have to go to sleep.,4703,私は眠らなければなりません。
2,1277,I have to go to sleep.,344904,そろそろ寝なくちゃ。
3,1280,Today is June 18th and it is Muiriel's birthday!,4705,今日は６月１８日で、ムーリエルの誕生日です！
4,1282,Muiriel is 20 now.,4707,ムーリエルは２０歳になりました。


In [13]:
# Dataset overview
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print()

# Missing/empty checks
empty_english = (df["english"].str.strip() == "").sum()
empty_japanese = (df["japanese"].str.strip() == "").sum()
print("Empty text counts:")
print(f"- english:  {empty_english:,}")
print(f"- japanese: {empty_japanese:,}")
print()

# Duplicate pair checks
duplicate_pairs = df.duplicated(subset=["english", "japanese"]).sum()
print(f"Duplicate EN-JP pairs: {duplicate_pairs:,}")
print()

# Length diagnostics (characters)
length_stats = df.assign(
    english_chars=df["english"].str.len(),
    japanese_chars=df["japanese"].str.len(),
)[["english_chars", "japanese_chars"]].describe(percentiles=[0.5, 0.9, 0.99])
print("Character length summary:")
display(length_stats)

# Quick sample
print("Sample rows:")
display(df.sample(5, random_state=42))

Rows: 279,694
Columns: ['eng_id', 'english', 'jp_id', 'japanese']

Empty text counts:
- english:  0
- japanese: 0

Duplicate EN-JP pairs: 2

Character length summary:


,english_chars,japanese_chars
count,279694.000000,279694.000000
mean,37.889168,17.210712
std,32.721087,7.997025
min,3.000000,2.000000
50%,34.000000,16.000000
90%,57.000000,26.000000
99%,107.000000,44.000000
max,13653.000000,302.000000


Sample rows:


,eng_id,english,jp_id,japanese
27874,41423,Don't keep company with such a bad boy.,204181,そんな悪い子と友達になるな。
216883,9354007,It wasn't our fault. Believe me.,9678940,私達のせいじゃないわ。信じてよ。
57687,71054,May I ask you a question?,149509,質問をしてもいいですか。
86982,264505,"In his autobiography, he repeatedly refers to ...",150052,自伝の中で彼はくりかえし不幸な少年時代に言及している。
254190,9193501,The apple cake's ran out.,11563342,アップルケーキがなくなっちゃった。


In [14]:
df.head(5)

,eng_id,english,jp_id,japanese
0,1276,Let's try something.,4702,何かしてみましょう。
1,1277,I have to go to sleep.,4703,私は眠らなければなりません。
2,1277,I have to go to sleep.,344904,そろそろ寝なくちゃ。
3,1280,Today is June 18th and it is Muiriel's birthday!,4705,今日は６月１８日で、ムーリエルの誕生日です！
4,1282,Muiriel is 20 now.,4707,ムーリエルは２０歳になりました。


## PDF Chunking for RAG Knowledge Base

This section processes two PDF documents into chunked `.jsonl` files for a RAG knowledge base:

1. **JTF Style Guide** (`jtf_style_guide_e.pdf`) — chunked by numbered section headings
2. **Tae Kim's Grammar Guide** (`grammar_guide.pdf`) — selected sections only: particle usage (は/も/が), polite vs plain form, transitive/intransitive verbs

Each chunk is a dict with fields: `source`, `section`, `text`, `chunk_id`. Outputs are saved to `data/kb/`.

In [15]:
%pip install -q pdfplumber

import pdfplumber
import json
import re
from pathlib import Path

KB_DIR = Path("../kb")
KB_DIR.mkdir(parents=True, exist_ok=True)


def extract_pdf_text(pdf_path):
    """Extract text from every page of a PDF, returned as one string."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n".join(pages)


def clean_pdf_text(text):
    """Clean common PDF extraction artifacts.

    Full-width numbers/letters are preserved — in these documents
    they are intentional examples, not extraction errors.
    """
    text = re.sub(r"--\s*\d+\s+of\s+\d+\s*--", "", text)
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "--")
    text = text.replace("\x0c", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n\d{1,3}\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_sentences(text):
    """Split mixed EN/JP text into sentence-level units."""
    paragraphs = re.split(r"\n\s*\n", text.strip())
    sentences = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        sents = re.split(
            r"(?<=[。！？])\s*(?=\S)|(?<=[.!?])\s+(?=[A-Z\"'(])",
            para,
        )
        sentences.extend(s.strip() for s in sents if s and s.strip())
    return sentences


def chunk_with_overlap(sentences, max_sents=5, overlap=2):
    """Group sentences into overlapping chunks."""
    if not sentences:
        return []
    if len(sentences) <= max_sents:
        return [" ".join(sentences)]
    chunks, step = [], max(1, max_sents - overlap)
    for i in range(0, len(sentences), step):
        chunk_sents = sentences[i : i + max_sents]
        chunks.append(" ".join(chunk_sents))
        if i + max_sents >= len(sentences):
            break
    return chunks


def estimate_tokens(text):
    """Rough token count (~4 chars / token for mixed EN/JP)."""
    return max(1, len(text) // 4)


def save_jsonl(records, path):
    """Write list of dicts as a .jsonl file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} chunks → {path}")


def print_summary(chunks, label):
    """Print chunk count, average token length, and sample chunks."""
    toks = [estimate_tokens(c["text"]) for c in chunks]
    avg = sum(toks) / len(toks) if toks else 0
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Total chunks : {len(chunks)}")
    print(f"  Avg length   : {avg:.0f} tokens (est.)\n")
    for c in chunks[:3]:
        preview = c["text"][:300] + ("…" if len(c["text"]) > 300 else "")
        print(f"  [{c['chunk_id']}]  section: {c['section']}")
        print(f"  {preview}\n")

Note: you may need to restart the kernel to use updated packages.


### 1. JTF Style Guide (`jtf_style_guide_e.pdf`)

Extract text with `pdfplumber`, split by numbered section headings (e.g. "1.1", "2.1.6"), and create sentence-level chunks with 1–2 sentence overlap. Each section becomes one or more chunks depending on its length.

In [17]:
raw_style = extract_pdf_text("../kb/jtf_style_guide_e.pdf")
style_text = clean_pdf_text(raw_style)

# Match section headings like "1.1. Style", "2.1.6. Chōon ...", etc.
# Requires at least one dot (e.g. "1.1") to avoid matching numbered list items.
heading_re = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.+?)$", re.MULTILINE)
matches = list(heading_re.finditer(style_text))
print(f"Found {len(matches)} section headings in JTF style guide\n")

style_chunks = []
source_style = "jtf_style_guide_e.pdf"

for idx, m in enumerate(matches):
    sec_num = m.group(1)
    sec_title = m.group(2).strip()
    heading = f"{sec_num} {sec_title}"

    start = m.start()
    end = matches[idx + 1].start() if idx + 1 < len(matches) else len(style_text)
    body = style_text[start:end].strip()

    sents = split_sentences(body)
    sub_chunks = chunk_with_overlap(sents, max_sents=5, overlap=2)

    for j, ct in enumerate(sub_chunks):
        style_chunks.append({
            "source": source_style,
            "section": heading,
            "text": ct,
            "chunk_id": f"jtf_{sec_num.replace('.', '_')}_{j:02d}",
        })

save_jsonl(style_chunks, KB_DIR / "style_guide_chunks.jsonl")
print_summary(style_chunks, "JTF Style Guide — Chunking Summary")

Found 143 section headings in JTF style guide

Saved 181 chunks → ..\kb\style_guide_chunks.jsonl

  JTF Style Guide — Chunking Summary
  Total chunks : 181
  Avg length   : 63 tokens (est.)

  [jtf_12_5_00]  section: 12.5 12．5
  12.5 12．5
single-byte comma (,)
12. Be consistent in unit nomenclature. Acceptable Unacceptable See
2kg 354m 20℃ 3ft 23坪 14km 5. Writing Units
Is consistently metric. Mixes American, Japanese, and
metric units.

  [jtf_12_5_01]  section: 12.5 12．5
  Writing Units
Is consistently metric. Mixes American, Japanese, and
metric units. Table of Contents
Introduction
1. Sentences ···················································································· 9

  [jtf_1_1_00]  section: 1.1 Style ................................................................................................... 9
  1.1. Style ................................................................................................... 9



### 2. Tae Kim's Grammar Guide (`grammar_guide.pdf`) — Selected Sections

Extract only three targeted topics from the 353-page guide:

- **Particle usage (は/も/が)** — sections 3.3 through 3.3.4
- **Polite vs plain form (です/ます vs だ/である)** — sections 4.1 through 4.1.5
- **Transitive & intransitive verbs** — sections 3.9 through 3.9.1

Same chunking strategy: sentence-level splits with 1–2 sentence overlap.

In [18]:
TARGET_SECTIONS = {
    "3.3", "3.3.1", "3.3.2", "3.3.3", "3.3.4",      # Particles は/も/が
    "3.9", "3.9.1",                                     # Transitive / intransitive
    "4.1", "4.1.1", "4.1.2", "4.1.3", "4.1.4", "4.1.5", # Polite form
}

raw_gram = extract_pdf_text("../kb/grammar_guide.pdf")
gram_text = clean_pdf_text(raw_gram)

# Remove repeating page headers like "3.3. INTRO TO PARTICLES CHAPTER 3. BASIC GRAMMAR"
# These contain section numbers that would confuse heading detection.
gram_text = re.sub(r"^.*CHAPTER \d+\..*$", "", gram_text, flags=re.MULTILINE)

heading_re = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.+?)$", re.MULTILINE)
all_matches = list(heading_re.finditer(gram_text))
print(f"Found {len(all_matches)} total section headings in grammar guide")

grammar_chunks = []
source_gram = "grammar_guide.pdf"
kept = 0

for idx, m in enumerate(all_matches):
    sec_num = m.group(1)
    if sec_num not in TARGET_SECTIONS:
        continue
    kept += 1
    sec_title = m.group(2).strip()
    heading = f"{sec_num} {sec_title}"

    start = m.start()
    end = all_matches[idx + 1].start() if idx + 1 < len(all_matches) else len(gram_text)
    body = gram_text[start:end].strip()

    sents = split_sentences(body)
    sub_chunks = chunk_with_overlap(sents, max_sents=5, overlap=2)

    for j, ct in enumerate(sub_chunks):
        grammar_chunks.append({
            "source": source_gram,
            "section": heading,
            "text": ct,
            "chunk_id": f"gram_{sec_num.replace('.', '_')}_{j:02d}",
        })

print(f"Kept {kept} sections matching target topics\n")
save_jsonl(grammar_chunks, KB_DIR / "grammar_chunks.jsonl")
print_summary(grammar_chunks, "Grammar Guide (Selected Sections) — Chunking Summary")

Found 570 total section headings in grammar guide
Kept 26 sections matching target topics

Saved 81 chunks → ..\kb\grammar_chunks.jsonl

  Grammar Guide (Selected Sections) — Chunking Summary
  Total chunks : 81
  Avg length   : 97 tokens (est.)

  [gram_3_3_00]  section: 3.3 Introduction to Particles . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 33
  3.3 Introduction to Particles . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 33

  [gram_3_3_1_00]  section: 3.3.1 Defining grammatical functions with particles . . . . . . . . . . . . . . . . 33
  3.3.1 Defining grammatical functions with particles . . . . . . . . . . . . . . . . 33

  [gram_3_3_2_00]  section: 3.3.2 The 「は」 topic particle . . . . . . . . . . . . . . . . . . . . . . . . . . . 33
  3.3.2 The 「は」 topic particle . . . . . . . . . . . . . . . . . . . . . . . . . . . 33

